In [2]:
!pip install boto3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import boto3
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker import Session
import io
from sagemaker.amazon.common import smac
import os
from sagemaker.amazon.amazon_estimator import get_image_uri

In [4]:
!curl -o student_scores.csv https://raw.githubusercontent.com/henrykohl/MLOps-Foundation/refs/heads/main/10.Segamaker/student_scores.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   214  100   214    0     0    566      0 --:--:-- --:--:-- --:--:--   566


In [5]:
df = pd.read_csv("student_scores.csv")
df.head()

,Hours,Scores
0,2.5,21
1,5.1,47
2,3.2,27
3,8.5,75
4,3.5,30


In [6]:
df.shape

(25, 2)

In [8]:
# separate x and y
x=df[["Hours"]]   ## 輸出 DataFrame
y=df[["Scores"]]  ## 輸出 DataFrame

In [9]:
# checking data types
x.dtypes
x = x.astype("float32")
y = y.astype("float32")

In [10]:
y.dtypes

,0
Scores,float32


In [11]:
# split the data
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

In [12]:
# reset index
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [13]:
X_train

,Hours
0,7.7
1,2.7
2,3.3
3,3.8
4,5.9
5,6.1
6,5.5
7,3.5
8,4.8
9,1.9


In [14]:
#we need to take label column as vector
y_train = y_train.iloc[:,0] # 輸出 Series

In [15]:
y_train

,Scores
0,85.0
1,30.0
2,42.0
3,35.0
4,62.0
5,67.0
6,60.0
7,30.0
8,54.0
9,24.0


In [16]:
y_test = y_test.iloc[:,0] # 輸出 Series

In [23]:
y_test

,Scores
0,81.0
1,41.0
2,20.0
3,30.0
4,75.0


# 建立 AWS s3 bucket

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_purposebuckets.png?raw=1" width=100%><br>
- 進入 Buckets 頁面，點選 Create bucket

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_createbuckett.png?raw=1" width=85%><br>
- Bucket name：bappy-bucket-25

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_createbucketb.png?raw=1" width=85%><br>
- 頁面下拉，然後建立

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_purposebuckets2.png?raw=1" width=85%><br>
- 回到 Buckets 頁面，點選剛建立 Bucket。

# AWS Sagemaker

- 此部分只能在 AWS 上運行，因為 Sagemaker 是 AWS 的產品

In [ ]:
#Lets create sagemaker session
sagemaker_session = sagemaker.Session()
#define the bucket name
bucket_name = "bappy-sagemaker"
#define the prefix
prefix = "linear-learner"
#get the execution role
role = sagemaker.get_execution_role()

In [ ]:
#convert X_train into numpy array
X_train = np.array(X_train)

In [ ]:
#create the buffer

buf=io.BytesIO() # 建立一個記憶體緩衝區

# buf: 目標檔案對象（通常是 BytesIO 或實體檔案）
# X_train: 你的 NumPy 陣列（特徵矩陣）
# y_train: 對應的標籤 NumPy 陣列（選填）
smac.write_numpy_to_dense_tensor(buf,X_train,y_train) # 將 NumPy 數據寫入緩衝區並轉換為 Dense Tensor 格式

buf.seek(0) # 指標回到緩衝區開頭，準備讀取上傳

In [ ]:
#define the name of the file
key="student-data"

#code to upload in s3
boto3.resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'train',key)).upload_fileobj(buf)

#path of our data
s3_train_data=f"s3://{bucket_name}/{prefix}/train/{key}"

print("Data uploaded",s3_train_data)

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_bucketdashboard.png?raw=1" width=90%><br>
- 到 AWS bucket 頁面，點選物件 linear-learner

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_bucketllobject.png?raw=1" width=90%><br>
- 點選 train 資料夾

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_buckettraindir.png?raw=1" width=90%><br>
- 檢視 train 資料夾

In [ ]:
#convert X_test to numpy array
X_test = np.array(X_test)

#create the buffer

buf=io.BytesIO() # 建立一個記憶體緩衝區

smac.write_numpy_to_dense_tensor(buf,X_test,y_test) # 將 NumPy 數據寫入緩衝區並轉換為 Dense Tensor 格式

buf.seek(0) # 指標回到緩衝區開頭，準備讀取上傳

#define the name of the file
key="student-data-test"

#code to upload in s3
boto3.resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'test',key)).upload_fileobj(buf)

#path of our data
s3_test_data=f"s3://{bucket_name}/{prefix}/test/{key}"

print("Data uploaded",s3_test_data)

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_bucketllobject2.png?raw=1" width=90%><br>
- 點選 test 資料夾

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_buckettestdir.png?raw=1" width=90%><br>
- 檢視 test 資料夾

In [ ]:
#output location
output_location = f"s3://{bucket_name}/{prefix}/output"
output_location

In [ ]:
#bring the container
container=sagemaker.image_uris.retrieve("linear-learner",boto3.Session().region_name)

In [ ]:
#define the estimator
linear=sagemaker.estimator.Estimator(container,role,instance_count=1,instance_type='ml.c4.xlarge',output_path=output_location,sagemaker_session=sagemaker_session)

In [ ]:
#setting up the hyper parameters
linear.set_hyperparameters(feature_dim=1,predictor_type="regressor",mini_batch_size=4,epochs=6,num_models=32,loss="absolute_loss")

In [ ]:
#fit the model (此步驟需花多些時間)
linear.fit({"train":s3_train_data})

In [ ]:
#deploy the model
linear_regresor=linear.deploy(initial_instance_count=1,instance_type="ml.m4.xlarge")

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_sagemakerinstance.png?raw=1" width=95%><br>
- 在 SageMaker AI 的 dashboard 中，inference 的 Endpoints 下，可以檢視部署完成的 endpoint。
- 越新的，列在越下方。

In [ ]:
linear_regresor.serializer=sagemaker.serializers.CSVSerializer()
linear_regresor.deserializer=sagemaker.deserializers.JSONDeserializer()

In [ ]:
#prediction
results=linear_regresor.predict(X_test)

In [ ]:
results

results 的樣態如下：
<pre>
{
    'predictions':[{'score':37.19...},
        {'score':24.09...},
        {'score':63.41...},
        ...
    ]
}
</pre>

In [ ]:
predictions=np.array([i["score"] for i in results["predictions"]])

In [ ]:
predictions

In [ ]:
linear_regresor.delete_endpoint()

<img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_sagemakerinstance2.png?raw=1" width=100%><br>
- 在 SageMaker AI 的 dashboard 中，inference 的 Endpoints 下，可以檢視之前部署的 endpoint 已經被刪除了。

* 若要刪除 Sagemaker 中的 instance

    <img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_instancestopspace.png?raw=1" width=100%><br>
    - 在 Sagemaker dashboard 中點選 Stop space

    <img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_instancestopspacec.png?raw=1" width=75%><br>
    - 按下 Stop space 以確認

    <img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_instancestopspaced.png?raw=1" width=100%><br>
    - 再回到 Sagemaker dashboard 中，點選 ⋮ 後，點選 Delete space

    <img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_jupyterlabspace.png?raw=1" width=100%><br>
    - 刪除 space 後的的頁面

* 刪除 bucket

  <img src="https://github.com/henrykohl/MLOps-Foundation/blob/main/figures/sagemaker/aws/aws_bucketdelete.png?raw=1" width=100%><br>
  - 點選欲刪除的 bucket，點選 delete
  - 注意，bucket 內必須是空的，若有物件存在，必須先刪除，否則 bucket 無法刪除